In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

In [ ]:
gold_dim_persoon = query_df("""
  SELECT p.id, p.titels, p.roepnaam, p.achternaam, p.overlijdensdatum, p.woonplaats, p.land, fzp.van, fzp.tot_en_met, f.afkorting
  FROM silver.persoon as p, silver.fractie_zetel_persoon as fzp, silver.fractie as f, silver.fractie_zetel as fz
  WHERE p.id = fzp.persoon_id
  AND fzp.fractie_zetel_id = fz.id
  AND fz.fractie_id = f.id
""")

gold_dim_persoon

In [ ]:
gold_fact_geschenk = query_df("""
  WITH gold_persoon AS (
  SELECT p.id, p.titels, p.roepnaam, p.achternaam, p.overlijdensdatum, p.woonplaats, p.land, fzp.van, fzp.tot_en_met, f.afkorting
  FROM silver.persoon as p, silver.fractie_zetel_persoon as fzp, silver.fractie as f, silver.fractie_zetel as fz
  WHERE p.id = fzp.persoon_id
  AND fzp.fractie_zetel_id = fz.id
  AND fz.fractie_id = f.id
  )
  SELECT g.id, g.persoon_id, p.roepnaam, p.achternaam, g.datum, g.gewicht, g.omschrijving, p.afkorting, p.overlijdensdatum
  FROM silver.persoon_geschenk as g
  INNER JOIN gold_persoon as p ON
  p.id = g.persoon_id
  AND g.datum BETWEEN p.van AND COALESCE(p.tot_en_met, CURRENT_DATE)
""")

gold_fact_geschenk

In [ ]:
gold_fact_reis = query_df("""
  WITH gold_persoon AS (
  SELECT p.id, p.titels, p.roepnaam, p.achternaam, p.overlijdensdatum, p.woonplaats, p.land, fzp.van, fzp.tot_en_met, f.afkorting
  FROM silver.persoon as p, silver.fractie_zetel_persoon as fzp, silver.fractie as f, silver.fractie_zetel as fz
  WHERE p.id = fzp.persoon_id
  AND fzp.fractie_zetel_id = fz.id
  AND fz.fractie_id = f.id
  )
  SELECT r.id, r.persoon_id, p.roepnaam, p.achternaam, r.doel, r.bestemming, r.van, r.tot_en_met, r.betaald_door, p.afkorting as fractielabel, SUM(r.tot_en_met - r.van) as reisduur, p.overlijdensdatum
  FROM silver.persoon_reis as r
  INNER JOIN gold_persoon as p ON
  p.id = r.persoon_id
  AND r.van BETWEEN p.van AND COALESCE(p.tot_en_met, CURRENT_DATE)
  GROUP BY r.id, r.persoon_id, p.roepnaam, p.achternaam, r.doel, r.bestemming, r.van, r.tot_en_met, r.betaald_door, fractielabel, p.overlijdensdatum
""")

gold_fact_reis

In [ ]:
# Export CSV

gold_fact_geschenk.to_csv("gold_fact_geschenk.csv")
gold_fact_reis.to_csv("gold_fact_reis.csv")